# vessel-watch: the classification study on per-vessel crops

Launcher only; all logic lives in the `vesselwatch` package. Two parts:

1. Rescore the nine original weights with one shared class index, to see whether the original numbering distorted the 2025 results.
2. Train the small CNN, the VGG-style CNN and the fine-tuned ResNet101 at label levels 1, 2 and 3 on per-vessel crops, select each on validation and score it once on the held-out test split.

Runtime > Change runtime type > GPU before running. If the session drops, run all cells again: finished runs are skipped.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Settings

In [ ]:
REPO_URL = 'https://github.com/bprtdaniel/vessel-watch.git'
BRANCH = 'main'

MODELS_DIR = '/content/drive/MyDrive/5.Projects/models'
DATA_DIR_DRIVE = '/content/drive/MyDrive/5.Projects/Data/ShipRSImageNet_V1'
DATA_DIR_LOCAL = '/content/data/ShipRSImageNet_V1'
RUNS_DIR = '/content/drive/MyDrive/5.Projects/runs'  # results are written straight to Drive
CACHE_DIR = '/content/crops'                         # extracted vessel crops, rebuilt each session
NUM_WORKERS = 8

## Code

In [ ]:
import os, subprocess

REPO_DIR = '/content/vessel-watch'
if os.path.exists(REPO_DIR):
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=True)
else:
    subprocess.run(['git', 'clone', '--branch', BRANCH, REPO_URL, REPO_DIR], check=True)

# Colab already ships torch, torchvision, pandas, scikit-learn, matplotlib and pyyaml
%pip install -q --no-deps -e /content/vessel-watch
%cd /content/vessel-watch
!git log --oneline -1

## Data

Reading images from Drive during training is slow, so they are copied to the Colab disk once per session.

In [ ]:
import shutil

if os.path.exists(DATA_DIR_LOCAL):
    print('Local copy already exists')
else:
    os.makedirs(os.path.dirname(DATA_DIR_LOCAL), exist_ok=True)
    shutil.copytree(DATA_DIR_DRIVE, DATA_DIR_LOCAL)
    print('Copied dataset to', DATA_DIR_LOCAL)

os.environ['VESSELWATCH_DATA'] = DATA_DIR_LOCAL
os.environ['VESSELWATCH_RUNS'] = RUNS_DIR
os.environ['VESSELWATCH_CACHE'] = CACHE_DIR

## 1. Did the original class numbering distort the scores?

The original study numbered the classes separately for train and val. This rescoring uses the train numbering for both. A run whose score stays the same was unaffected; a run that differs was scored against shifted labels.

In [ ]:
!python -m vesselwatch.verify_legacy --models-dir "$MODELS_DIR" --shared-classes --num-workers $NUM_WORKERS --out "$MODELS_DIR/legacy_verification_shared_classes.json"

## 2. The study on per-vessel crops

Nine runs, one after the other. Each writes its checkpoints, history, `metrics_val.json`, `metrics_test.json` and confusion matrices to its own folder under `RUNS_DIR`; the table at the end is also saved as `study_summary.json`.

In [ ]:
!python -m vesselwatch.study --configs configs/crops --num-workers $NUM_WORKERS